In [ ]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

# `ArrayList` vs. `LinkedList`

| operation | `ArrayList` | `LinkedList` |
|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(N)` |
| `add(e)` (back) | `O(1)` amortized | `O(1)` |
| `addFirst(e)`, `add(0, e)` | `O(N)` | `O(1)` |
| `removeLast()`, `remove(size - 1)` | `O(1)` | `O(1)` |
| `removeFirst()`, `remove(0)` | `O(N)` | `O(1)` |
| `add(i, e)`, `remove(i)` (middle) | `O(N)` | `O(N)` |
| `remove` / `add` through an iterator | `O(N)` | `O(1)` |
| `indexOf`, `contains` | `O(N)` | `O(N)` |
| memory per element | one array cell | one node: data, two references, object header |

Inserting in the middle is `O(N)` for both. Which one is faster?

In [ ]:
System.out.printf("%8s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 10_000; n <= 40_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        List<Integer> l = new ArrayList<>();
        for (int i = 0; i < N; i++) l.add(l.size() / 2, i);
    });
    long lnk = millis(() -> {
        List<Integer> l = new LinkedList<>();
        for (int i = 0; i < N; i++) l.add(l.size() / 2, i);
    });
    System.out.printf("%,8d %10d %10d%n", N, arr, lnk);
}

- Same `O(N)`: where does the difference come from? Shifting a block of consecutive cells vs. following `N/2` references to nodes scattered in memory.
- In practice: `ArrayList` is the default choice. `LinkedList` when the work happens at the ends, or through an iterator.

# `String` vs. `StringBuilder`

## `String` is immutable

- No method changes a `String`: `concat`, `toUpperCase`, `+` return a **new** one.
- `s += x`: a new string, all of `s` copied, then `x`. Cost: `O(length of s)`.

```java
String s = "";
for (int i = 0; i < N; i++)
    s += i % 10;        // copies the i characters built so far
```

Cost of the loop?

## `StringBuilder`

- An array of characters with spare capacity, plus a count, so basically an ArrayList for characters
- Full: a new array of about double the capacity, copy
- `append`: `O(1)` amortized. `toString()`: one copy at the end, `O(N)`
- Also: `insert(pos, x)`, `deleteCharAt(pos)`, `setCharAt(pos, c)`, `reverse()`, `length()`

In [ ]:
StringBuilder sb = new StringBuilder("list");
sb.append(": ").append(3.5).append(", ").append(true);     // append returns sb itself
System.out.println(sb);

sb.insert(0, '[').append(']');
sb.setCharAt(1, 'L');
System.out.println(sb + " " + sb.length());

In [ ]:
System.out.printf("%8s %10s %10s%n", "N", "+= ms", "append ms");
for (int n = 25_000; n <= 200_000; n *= 2) {
    final int N = n;
    long str = millis(() -> {
        String s = "";
        for (int i = 0; i < N; i++) s += i % 10;
    });
    long bld = millis(() -> {
        StringBuilder b = new StringBuilder();
        for (int i = 0; i < N; i++) b.append(i % 10);
        String s = b.toString();
    });
    System.out.printf("%,8d %10d %10d%n", N, str, bld);
}

- Which column grows ×4 when `N` doubles?
- `"x = " + x + ", y = " + y` in one expression: a problem?
- `sb.insert(0, c)` in a loop of `N` iterations: cost?
- `toString` in `MyLinkedListDouble`: why a `StringBuilder`?

# Stacks

## Stack ADT

- Elements are added and removed at the **same end**: the **top**.
- **LIFO**: last in, first out.
- Examples: undo in an editor, back button in a browser, the call stack.

| operation | |
|---|---|
| `push(e)` | add `e` on top |
| `pop()` | remove and return the top element |
| `peek()` | return the top element, without removing it |
| `isEmpty()`, `size()` | |

- No `get(i)`, no `add(i, e)`: only the top is accessible.
- A list can do everything a stack does. Why define a separate ADT?

In [ ]:
interface MyStack<E> {
    void push(E e);

    // throw NoSuchElementException if the stack is empty
    E pop();
    E peek();

    boolean isEmpty();
    int size();
}

### Popping an empty stack

- Our choice: throw `NoSuchElementException`.
- Alternative: return `null`. What happens after `push(null)`?

## Array-based stack

- Elements in `data[0..size-1]`, as in `MyArrayListDouble`.
- Where is the top: index `0` or index `size - 1`? Cost of `push` and `pop` in each case?

Top at `size - 1`:
- next free cell: `data[size]`
- top element: `data[size - 1]`
- full array: double the capacity

### Generic array

- `new E[4]` does not compile (erasure).
- `(E[]) new Object[4]` instead, with `@SuppressWarnings("unchecked")`.

In [ ]:
class ArrayStack<E> implements MyStack<E> {
    private E[] data;
    private int size;       // elements in data[0..size-1], top at size-1

    @SuppressWarnings("unchecked")
    ArrayStack() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    public void push(E e) {
        if (size == data.length)
            data = Arrays.copyOf(data, 2 * data.length);
        data[size] = e;
        size++;
    }

    public E pop() {
        if (size == 0) throw new NoSuchElementException("empty stack");
        size--;
        E e = data[size];
        data[size] = null;
        return e;
    }

    public E peek() {
        if (size == 0) throw new NoSuchElementException("empty stack");
        return data[size - 1];
    }

    // bottom to top: the top is the rightmost element
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[i]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }
}

In [ ]:
MyStack<String> s = new ArrayStack<>();
s.push("a");
s.push("b");
s.push("c");
System.out.println(s + "  peek=" + s.peek() + "  size=" + s.size());

for (String x : List.of("d", "e", "f")) s.push(x);     // grows from 4 to 8
System.out.println(s);

while (!s.isEmpty())
    System.out.print(s.pop() + " ");
System.out.println();

try {
    s.pop();
} catch (NoSuchElementException e) {
    System.out.println(e);
}

- `data[size] = null` in `pop`: the stack works without it. Why is it there?
  - `MyArrayListDouble` did not need it: why?
- Cost of `push`? Of `pop`, `peek`?

## Linked stack

- A chain of nodes, as in `MyLinkedListDouble`.
- Where is the top: first node or last node?

| top at | `push` | `pop` |
|---|---|---|
| first node (`head`) | ? | ? |
| last node (`tail`) | ? | ? |

Top at the first node:
- `push`: `addFirst`
- `pop`: `removeFirst`
- No `tail` needed. We call `head` `top`.

```java
public void push(E e) {
    top = new Node<>(e, top);       // the new node points to the old top
    size++;
}
```

Does it work on an empty stack?

In [ ]:
class LinkedStack<E> implements MyStack<E> {
    private static class Node<E> {
        E data;
        Node<E> next;       // the node below this one

        Node(E data, Node<E> next) {
            this.data = data;
            this.next = next;
        }
    }

    private Node<E> top;    // null if empty
    private int size;

    public int size() { return size; }
    public boolean isEmpty() { return top == null; }

    public void push(E e) {
        top = new Node<>(e, top);
        size++;
    }

    public E pop() {
        if (top == null) throw new NoSuchElementException("empty stack");
        E e = top.data;
        top = top.next;
        size--;
        return e;
    }

    public E peek() {
        if (top == null) throw new NoSuchElementException("empty stack");
        return top.data;
    }

    // top to bottom
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("top: ");
        for (Node<E> n = top; n != null; n = n.next)
            sb.append(n.data).append(" ");
        return sb.toString();
    }
}

In [ ]:
MyStack<Integer> t = new LinkedStack<>();
for (int i = 1; i <= 5; i++) t.push(i * i);
System.out.println(t + "  size=" + t.size());
System.out.println(t.pop() + " " + t.pop() + " " + t.peek());
System.out.println(t + "  size=" + t.size());

- Popped node: who frees it?
- Same `MyStack` interface: the code using the stack does not change. Only the `new` does.

## Array vs. linked stack

| | array | linked |
|---|---|---|
| `push` | `O(1)` amortized, `O(N)` worst case | `O(1)` |
| `pop`, `peek` | `O(1)` | `O(1)` |
| memory | one cell per element, plus unused capacity | one node per element |

In [ ]:
System.out.printf("%10s %10s %10s%n", "N", "array ms", "linked ms");
for (int n = 1_000_000; n <= 8_000_000; n *= 2) {
    final int N = n;
    long arr = millis(() -> {
        MyStack<Integer> s = new ArrayStack<>();
        for (int i = 0; i < N; i++) s.push(i % 10);    // small Integers are cached: no allocation
        while (!s.isEmpty()) s.pop();
    });
    long lnk = millis(() -> {
        MyStack<Integer> s = new LinkedStack<>();
        for (int i = 0; i < N; i++) s.push(i % 10);
        while (!s.isEmpty()) s.pop();
    });
    System.out.printf("%,10d %10d %10d%n", N, arr, lnk);
}

- Both columns roughly double with `N`: expected?
- Same `O(1)` per operation: where does the gap come from? Compare with `ArrayList` vs. `LinkedList` above.
- A program needs every `push` to take the same time (no occasional `O(N)`): which one?